# Lab 03 — Search by Meaning, Not Just Words

A pretrained model may create embeddings. Your job is to implement cosine similarity and deterministic top-*k* retrieval in NumPy.

**Rules:** CPU only; no vector database; no hosted embedding API; no library nearest-neighbour/search implementation.

In [1]:
%pip -q install sentence-transformers numpy

In [2]:
from time import perf_counter
import numpy as np
from sentence_transformers import SentenceTransformer

np.set_printoptions(precision=4, suppress=True)
RNG = np.random.default_rng(42)
MODEL_NAME = "all-MiniLM-L6-v2"

## Challenge corpus

The data is deterministic and deliberately contains paraphrases. It is lab-only data, not the InstaKill domain model.

In [3]:
TOPICS = {
    "food": [
        "A budget-friendly restaurant opened near the station.",
        "Looking for an affordable place to eat dinner tonight.",
        "The cheapest lunch menu I found this week costs under ten euros.",
    ],
    "running": [
        "Easy jogging route beside the river for beginners.",
        "My first 5K training plan starts with short runs and walks.",
        "Anyone know a calm place for an evening run?",
    ],
    "python": [
        "I finally understood why a Python dictionary needs hashable keys.",
        "Small NumPy experiment: vectorized code was much faster than my loop.",
        "Looking for a clear explanation of Python type hints.",
    ],
    "music": [
        "Quiet jazz playlist for a long study session.",
        "Any recommendations for live music this weekend?",
        "This new album is perfect background sound while reading.",
    ],
    "travel": [
        "Best low-cost train trip for a one-day escape?",
        "I need ideas for a weekend hike without a car.",
        "What should I pack for a short mountain trip in autumn?",
    ],
    "books": [
        "Recommend a short science-fiction novel with a strong ending.",
        "I want a book that explains machine learning without too much math.",
        "Which library has a good quiet corner for reading?",
    ],
}

def make_demo_posts(repetitions=200):
    posts = []
    post_id = 1
    for topic, templates in TOPICS.items():
        for repeat in range(repetitions):
            template = templates[repeat % len(templates)]
            posts.append({
                "post_id": post_id,
                "user_id": (repeat % 37) + 1,
                "topic": topic,
                "content": f"{template} [community note {repeat + 1}]",
            })
            post_id += 1
    RNG.shuffle(posts)
    return posts

posts = make_demo_posts()
texts = [post["content"] for post in posts]
assert len(posts) >= 1_000
print(f"{len(posts)} posts; example: {posts[0]}")

1200 posts; example: {'post_id': 70, 'user_id': 33, 'topic': 'food', 'content': 'A budget-friendly restaurant opened near the station. [community note 70]'}


## Baseline: keyword overlap

This baseline is intentionally simple. You may inspect it, but do not use it as your semantic solution.

In [4]:
def keyword_search(query, posts, k=5):
    query_terms = set(query.lower().split())
    scored = []
    for post in posts:
        score = len(query_terms & set(post["content"].lower().split()))
        scored.append((score, post["post_id"], post))
    scored.sort(key=lambda item: (-item[0], item[1]))
    return [post for score, _, post in scored[:k] if score > 0]

keyword_search("cheap dinner", posts, k=3)

[{'post_id': 2,
  'user_id': 2,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 2]'},
 {'post_id': 5,
  'user_id': 5,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 5]'},
 {'post_id': 8,
  'user_id': 8,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 8]'}]

## TODO 1 — Encode text

Load the supplied pretrained encoder and create a normalized embedding matrix for all corpus texts. Keep the model call at this boundary; your retrieval functions below must not load a model or perform I/O.

In [14]:

# Load the pretrained language model
model = SentenceTransformer(MODEL_NAME, device="cpu")

# Transform all texts into normalized numerical vectors
corpus_embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Ensure the matrix uses a floating-point numerical data type
corpus_embeddings = corpus_embeddings.astype(np.float32)

print("Number of posts:", len(posts))
print("Matrix dimension:", corpus_embeddings.shape)
print("Data type:", corpus_embeddings.dtype)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Number of posts: 1200
Matrix dimension: (1200, 384)
Data type: float32


## TODO 2 — Pure retrieval functions

Implement these functions with NumPy. A zero vector must not produce `NaN` or infinity. For equal scores, the lower `post_id` comes first.

In [15]:

def cosine_similarity(query_vector, corpus_vectors):
    """Return the cosine similarity score for each vector in the corpus"""

    query_vector = np.asarray(query_vector, dtype=float)
    corpus_vectors = np.asarray(corpus_vectors, dtype=float)

    # Calculate the dot product between the query and each post
    dot_products = corpus_vectors @ query_vector

    # Calculate the vector norms
    query_norm = np.linalg.norm(query_vector)
    corpus_norms = np.linalg.norm(corpus_vectors, axis=1)

    # Perform division only for non-zero norms
    denominators = corpus_norms * query_norm

    scores = np.zeros(len(corpus_vectors), dtype=float)

    np.divide(
        dot_products,
        denominators,
        out=scores,
        where=denominators > 0
    )

    return scores


def stable_top_k(scores, post_ids, k):
    """Return indices sorted by score and then by post_id"""

    scores = np.asarray(scores)
    post_ids = np.asarray(post_ids)

    if k <= 0:
        return np.array([], dtype=int)

    # Sort by descending score; in case of a tie, sort by ascending post_id
    indices = np.lexsort((post_ids, -scores))

    return indices[:k]


def semantic_search(query_embedding, posts, corpus_embeddings, k=5):
    """Return the most relevant posts and their scores"""

    scores = cosine_similarity(
        query_embedding,
        corpus_embeddings
    )

    post_ids = np.array([post["post_id"] for post in posts])

    indices = stable_top_k(scores, post_ids, k)

    results = []

    for index in indices:
        result = {
            "post_id": posts[index]["post_id"],
            "user_id": posts[index]["user_id"],
            "content": posts[index]["content"],
            "score": float(scores[index])
        }

        results.append(result)

    return results


## Acceptance checks

Run this cell only after implementing the TODOs. These checks are deliberately small; they verify the retrieval contract, not whether a model is magically perfect.

In [9]:
tiny_query = np.array([1.0, 0.0])
tiny_corpus = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.0], [0.0, 0.0]])
tiny_scores = cosine_similarity(tiny_query, tiny_corpus)
assert np.allclose(tiny_scores, [1.0, 0.0, 1.0, 0.0])
assert np.isfinite(cosine_similarity(np.array([0.0, 0.0]), tiny_corpus)).all()

tie_indices = stable_top_k(np.array([0.7, 0.9, 0.9]), np.array([7, 5, 2]), k=2)
assert list(tie_indices) == [2, 1], "ties must use ascending post_id"

assert corpus_embeddings.shape[0] == len(posts)
assert np.issubdtype(corpus_embeddings.dtype, np.floating)

query_embedding = model.encode(["cheap dinner near the station"], normalize_embeddings=True)[0]
results = semantic_search(query_embedding, posts, corpus_embeddings, k=5)
assert 1 <= len(results) <= 5
assert all({"post_id", "user_id", "content", "score"} <= set(result) for result in results)
assert all(results[i]["score"] >= results[i + 1]["score"] for i in range(len(results) - 1))
assert all(np.isfinite(result["score"]) for result in results)
print("Acceptance checks passed.")

Acceptance checks passed.


## Compare search behavior

Use at least three queries. For each, compare the keyword baseline and your semantic search. Record one useful result and one surprising or weak result.

In [16]:

for query in queries:
    print("\n" + "=" * 70)
    print("INTEROGARE:", query)

    # Keyword search
    keyword_results = keyword_search(query, posts, k=5)

    print("\n--- Keyword search ---")
    for post in keyword_results:
        print(f"ID: {post['post_id']} | {post['content']}")

    # Encode the query as an embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Semantic search
    semantic_results = semantic_search(
        query_embedding,
        posts,
        corpus_embeddings,
        k=5
    )

    print("\n--- Semantic search ---")
    for result in semantic_results:
        print(
            f"ID: {result['post_id']} | "
            f"Scor: {result['score']:.4f} | "
            f"{result['content']}"
        )



INTEROGARE: cheap dinner

--- Keyword search ---
ID: 2 | Looking for an affordable place to eat dinner tonight. [community note 2]
ID: 5 | Looking for an affordable place to eat dinner tonight. [community note 5]
ID: 8 | Looking for an affordable place to eat dinner tonight. [community note 8]
ID: 11 | Looking for an affordable place to eat dinner tonight. [community note 11]
ID: 14 | Looking for an affordable place to eat dinner tonight. [community note 14]

--- Semantic search ---
ID: 194 | Scor: 0.6392 | Looking for an affordable place to eat dinner tonight. [community note 194]
ID: 197 | Scor: 0.6358 | Looking for an affordable place to eat dinner tonight. [community note 197]
ID: 26 | Scor: 0.6346 | Looking for an affordable place to eat dinner tonight. [community note 26]
ID: 2 | Scor: 0.6319 | Looking for an affordable place to eat dinner tonight. [community note 2]
ID: 137 | Scor: 0.6315 | Looking for an affordable place to eat dinner tonight. [community note 137]

INTEROGARE:

## RESULTS

Replace every placeholder before submission.

```text
RESULTS
baseline:
solution:
runtime:
memory / model size:
3 failure cases:
what I would try next:
```